In [1]:
# Cellule 1: Changer de répertoire et charger le module
println("Re-loading NeuroDSL module after kernel fixes...")

# Important: on doit inclure depuis le dossier src/
include("src/NeuroDSL.jl")
using .NeuroDSL

println("--- Verification anti-duplication after reload ---")
checks = [
    (NeuroDSL.demand!,           "demand!",                    2),
    (NeuroDSL.backward_graph!,   "backward_graph!",            1),
    (NeuroDSL.execute_rule!,     "NeuroDSL.execute_rule!",     1),
    (NeuroDSL.accum_grad!,       "accum_grad!",                1),
]
all_ok = true
for (fn, name, max_m) in checks
    n = length(methods(fn))
    ok = n <= max_m
    global all_ok = all_ok && ok
    println(ok ? "✅" : "❌", " $name — $n méthode(s)")
end
if all_ok
    println("\n🚀 API chargée sans duplication.")
else
    error("❌ Duplication détectée !")
end

Re-loading NeuroDSL module after kernel fixes...
--- Verification anti-duplication after reload ---
✅ demand! — 1 méthode(s)
✅ backward_graph! — 1 méthode(s)
✅ NeuroDSL.execute_rule! — 1 méthode(s)
✅ accum_grad! — 1 méthode(s)

🚀 API chargée sans duplication.


In [2]:
using CUDA

# 1. Structure de nœud individuel persistant
mutable struct MnistNodeV5
    symbol::Symbol
    value::Union{Nothing, CuArray{Float32}}
    gradient::Union{Nothing, CuArray{Float32}}
    stream::CUDA.CuStream
end

# 2. Le Graphe global contenant les poids partagés et les deux emplacements alternés (slots)
mutable struct MnistParallelGraph
    weights::Dict{Symbol, CuArray{Float32}}  # Contient :W1, :W2, etc.
    batch_A::Dict{Symbol, MnistNodeV5}       # Slot de pipeline A
    batch_B::Dict{Symbol, MnistNodeV5}       # Slot de pipeline B
end

println("✅ Pas 1 réussi : Structures MnistNodeV5 et MnistParallelGraph définies avec succès.")

✅ Pas 1 réussi : Structures MnistNodeV5 et MnistParallelGraph définies avec succès.


In [3]:
# Fonction utilitaire pour instancier un dictionnaire de nœuds isolés sur un stream donné
function creer_slot_pipeline(symboles::Vector{Symbol}, stream::CUDA.CuStream)
    slot = Dict{Symbol, MnistNodeV5}()
    for sym in symboles
        # On initialise les valeurs et gradients à Nothing (ils seront alloués au premier warm-up)
        slot[sym] = MnistNodeV5(sym, nothing, nothing, stream)
    end
    return slot
end

# Fonction globale d'initialisation du graphe parallèle
function initialiser_graphe_mnist(d_in::Int, d_hidden::Int, d_out::Int)
    # 1. Allocation des poids partagés (W1, W2)
    weights = Dict{Symbol, CuArray{Float32}}()
    weights[:W1] = CUDA.randn(Float32, d_hidden, d_in) .* 0.1f0
    weights[:W2] = CUDA.randn(Float32, d_out, d_hidden) .* 0.1f0

    # Liste des symboles logiques qui vont transiter dans notre Transformer/Réseau
    # :x = entrée, :h = couche cachée, :pred = sortie, :y = cible, :loss = fonction de coût
    symboles_graphe = [:x, :h, :pred, :y, :loss]

    # 2. Création des deux streams matériels GPU distincts
    stream_A = CUDA.CuStream()
    stream_B = CUDA.CuStream()

    # 3. Instanciation des slots du pipeline avec leurs streams respectifs
    batch_A = creer_slot_pipeline(symboles_graphe, stream_A)
    batch_B = creer_slot_pipeline(symboles_graphe, stream_B)

    println("✅ Pas 2 réussi : Poids initialisés et Streams CUDA (A et B) alloués séparément.")
    return MnistParallelGraph(weights, batch_A, batch_B)
end

# Exemple de test d'initialisation (Entrée: 784, Cachée: 128, Sortie: 10)
graphe_test = initialiser_graphe_mnist(784, 128, 10)

✅ Pas 2 réussi : Poids initialisés et Streams CUDA (A et B) alloués séparément.


MnistParallelGraph(Dict{Symbol, CuArray{Float32}}(:W1 => [0.09766809 -0.105750374 … -0.10038996 -0.12050333; -0.09780806 -0.040567663 … 0.008204749 0.05648538; … ; 0.1610478 0.08853479 … -0.11205031 -0.061337736; -0.008402779 0.042030256 … 0.042706087 -0.04131229], :W2 => [-0.016976018 0.026029352 … 0.023912326 0.10947219; 0.07573592 0.17496471 … 0.051185053 -0.021865522; … ; 0.07149707 -0.0738014 … -0.07878142 -0.031681072; -0.0895907 -0.054716386 … -0.20205401 0.0036053676]), Dict{Symbol, MnistNodeV5}(:pred => MnistNodeV5(:pred, nothing, nothing, CuStream(0x000002632bb5df00, CuContext(0x000002630b0657e0))), :loss => MnistNodeV5(:loss, nothing, nothing, CuStream(0x000002632bb5df00, CuContext(0x000002630b0657e0))), :y => MnistNodeV5(:y, nothing, nothing, CuStream(0x000002632bb5df00, CuContext(0x000002630b0657e0))), :h => MnistNodeV5(:h, nothing, nothing, CuStream(0x000002632bb5df00, CuContext(0x000002630b0657e0))), :x => MnistNodeV5(:x, nothing, nothing, CuStream(0x000002632bb5df00, Cu

In [4]:
# Ré-exécution du Pas 4 avec signature flexible pour les données
function train_mnist_pipeline!(graph::MnistParallelGraph, x_data::Vector{<:CuArray}, y_data::Vector{<:CuArray}, lr::Float32)
    num_batches = length(x_data)
    if num_batches < 2
        error("Le pipeline nécessite au moins 2 batchs pour fonctionner.")
    end

    # 1. Allocation fixe et statique des buffers de gradients
    grads_accumulated = Dict{Symbol, CuArray{Float32}}()
    grads_accumulated[:W1] = CUDA.zeros(Float32, size(graph.weights[:W1])...)
    grads_accumulated[:W2] = CUDA.zeros(Float32, size(graph.weights[:W2])...)

    # Amorçage du pipeline sur le Slot A
    execute_forward_graph_mnist!(graph.batch_A, graph.weights, x_data[1], y_data[1])

    # 2. Boucle de pipeline asynchrone
    for i in 2:num_batches
        current_slot = (i % 2 == 0) ? graph.batch_A : graph.batch_B
        next_slot    = (i % 2 == 0) ? graph.batch_B : graph.batch_A

        X_next = x_data[i]
        Y_next = y_data[i]

        st_current = current_slot[:x].stream
        CUDA.stream!(st_current) do
            grads_accumulated[:W1] .= 0.0f0
            grads_accumulated[:W2] .= 0.0f0
        end

        # Parallélisme de flux (Backward N et Forward N+1)
        tache_backward = @async begin
            execute_backward_graph_mnist!(current_slot, graph.weights, grads_accumulated)
            CUDA.stream!(st_current) do
                graph.weights[:W1] .-= lr .* grads_accumulated[:W1]
                graph.weights[:W2] .-= lr .* grads_accumulated[:W2]
            end
        end

        tache_forward = @async begin
            execute_forward_graph_mnist!(next_slot, graph.weights, X_next, Y_next)
        end

        wait(tache_backward)
        wait(tache_forward)
        CUDA.synchronize()
    end

    # 3. Vidage du pipeline
    dernier_slot = (num_batches % 2 == 0) ? graph.batch_B : graph.batch_A
    st_dernier = dernier_slot[:x].stream

    execute_backward_graph_mnist!(dernier_slot, graph.weights, grads_accumulated)
    CUDA.stream!(st_dernier) do
        graph.weights[:W1] .-= lr .* grads_accumulated[:W1]
        graph.weights[:W2] .-= lr .* grads_accumulated[:W2]
    end

    CUDA.synchronize()
    println("✅ Pas 4 mis à jour : Signature assouplie et prête.")
end

train_mnist_pipeline! (generic function with 1 method)

In [5]:
using MLDatasets
using LinearAlgebra

function preparer_vrai_mnist(B::Int)
    println("📥 Chargement de MNIST depuis MLDatasets...")
    train_x, train_y = MNIST(split=:train)[:]

    # train_x est (28, 28, 60000). On va l'aplatir en (60000, 784)
    num_samples = size(train_x, 3)
    X_flat = zeros(Float32, num_samples, 784)
    for i in 1:num_samples
        X_flat[i, :] = vec(train_x[:, :, i])
    end

    # Encodage One-Hot des étiquettes (0-9 -> vecteur de taille 10)
    Y_onehot = zeros(Float32, num_samples, 10)
    for i in 1:num_samples
        label = train_y[i] # 0 à 9
        Y_onehot[i, label + 1] = 1.0f0
    end

    # Découpage en mini-batchs et transfert immédiat sur le GPU
    x_batches = [CuArray(X_flat[i:min(i+B-1, num_samples), :]) for i in 1:B:num_samples]
    y_batches = [CuArray(Y_onehot[i:min(i+B-1, num_samples), :]) for i in 1:B:num_samples]

    # On retire le dernier batch s'il n'est pas complet pour garder des tailles fixes
    if size(x_batches[end], 1) != B
        pop!(x_batches)
        pop!(y_batches)
    end

    println("✅ Données prêtes : $(length(x_batches)) batchs de taille $B créés en VRAM.")
    return x_batches, y_batches
end

# Initialisation des vrais batchs (Taille de batch = 128)
B_taille = 128
x_vrai, y_vrai = preparer_vrai_mnist(B_taille)

📥 Chargement de MNIST depuis MLDatasets...
✅ Données prêtes : 468 batchs de taille 128 créés en VRAM.


(CuArray{Float32, 2, CUDACore.DeviceMemory}[[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]  …  [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 

# Forward

In [6]:
# 1. FORWARD LOGIQUE ADAPTÉ À LA CLASSIFICATION
function execute_forward_graph_mnist!(slot::Dict{Symbol, MnistNodeV5}, weights::Dict{Symbol, CuArray{Float32}}, X_batch::CuArray{Float32}, Y_batch::CuArray{Float32})
    st = slot[:x].stream
    slot[:x].value = X_batch
    slot[:y].value = Y_batch

    # Couche 1 : Dense (h = X * W1')
    if isnothing(slot[:h].value)
        slot[:h].value = CUDA.zeros(Float32, size(X_batch, 1), size(weights[:W1], 1))
    end
    CUDA.stream!(st) do
        CUDA.CUBLAS.gemm!('N', 'T', 1.0f0, slot[:x].value, weights[:W1], 0.0f0, slot[:h].value)
    end

    # Activation : ReLU in-place
    CUDA.stream!(st) do
        slot[:h].value .= max.(0.0f0, slot[:h].value)
    end

    # Couche 2 : Dense Logits (pred = h * W2')
    if isnothing(slot[:pred].value)
        slot[:pred].value = CUDA.zeros(Float32, size(slot[:h].value, 1), size(weights[:W2], 1))
    end
    CUDA.stream!(st) do
        CUDA.CUBLAS.gemm!('N', 'T', 1.0f0, slot[:h].value, weights[:W2], 0.0f0, slot[:pred].value)
    end

    # Perte : Vraie Cross-Entropy (Correction du epsilon Float32)
    if isnothing(slot[:loss].value)
        slot[:loss].value = CUDA.zeros(Float32, 1)
    end
    CUDA.stream!(st) do
        max_val = maximum(slot[:pred].value, dims=2)
        exps = exp.(slot[:pred].value .- max_val)
        probs = exps ./ sum(exps, dims=2)
        # Utilisation de 1.0f-8 qui est la syntaxe Float32 officielle et robuste
        slot[:loss].value .= -sum(slot[:y].value .* log.(probs .+ 1.0f-8)) / size(X_batch, 1)
    end
end

# 2. BACKWARD LOGIQUE ADAPTÉ À LA CLASSIFICATION
function execute_backward_graph_mnist!(slot::Dict{Symbol, MnistNodeV5}, weights::Dict{Symbol, <:CuArray}, grads_accumulated::Dict{Symbol, <:CuArray})
    st = slot[:x].stream
    N = size(slot[:x].value, 1)

    # Raccourci analytique : dLoss/dPred = (Probs - Y) / N
    if isnothing(slot[:pred].gradient)
        slot[:pred].gradient = CUDA.zeros(Float32, size(slot[:pred].value)...)
    end
    CUDA.stream!(st) do
        max_val = maximum(slot[:pred].value, dims=2)
        exps = exp.(slot[:pred].value .- max_val)
        probs = exps ./ sum(exps, dims=2)
        slot[:pred].gradient .= (probs .- slot[:y].value) ./ Float32(N)
    end

    # Gradient par rapport à W2
    CUDA.stream!(st) do
        CUDA.CUBLAS.gemm!('T', 'N', 1.0f0, slot[:pred].gradient, slot[:h].value, 1.0f0, grads_accumulated[:W2])
    end

    # Rétropropagation vers H (dH = dPred * W2)
    if isnothing(slot[:h].gradient)
        slot[:h].gradient = CUDA.zeros(Float32, size(slot[:h].value)...)
    end
    CUDA.stream!(st) do
        CUDA.CUBLAS.gemm!('N', 'N', 1.0f0, slot[:pred].gradient, weights[:W2], 0.0f0, slot[:h].gradient)
        # Dérivée de la ReLU in-place
        slot[:h].gradient .= slot[:h].gradient .* (slot[:h].value .> 0.0f0)
    end

    # Gradient par rapport à W1
    CUDA.stream!(st) do
        CUDA.CUBLAS.gemm!('T', 'N', 1.0f0, slot[:h].gradient, slot[:x].value, 1.0f0, grads_accumulated[:W1])
    end
end

println("✅ Signature assouplie et prête pour le dispatch dynamique.")

println("✅ Pas 3 corrigé avec succès.")

✅ Signature assouplie et prête pour le dispatch dynamique.
✅ Pas 3 corrigé avec succès.


In [7]:
function calculer_accuracy(slot::Dict{Symbol, MnistNodeV5})
    # Récupérer les prédictions et les cibles
    pred = Array(slot[:pred].value)  # shape (batch, 10)
    y = Array(slot[:y].value)        # shape (batch, 10)
    
    # Pour chaque exemple, trouver la classe prédite (argmax) et la classe réelle
    correct = sum(argmax(pred, dims=2) .== argmax(y, dims=2))
    total = size(pred, 1)
    return (correct / total) * 100.0
end

calculer_accuracy (generic function with 1 method)

In [8]:
# 1. Configuration des dimensions pour le vrai MNIST
D_in = 784      # 28x28 pixels
D_hidden = 128  # 128 neurones cachés
D_out = 10      # 10 classes (chiffres de 0 à 9)

# 2. Initialisation d'un graphe tout neuf avec des poids aléatoires calibrés
graphe_vrai_mnist = initialiser_graphe_mnist(D_in, D_hidden, D_out)

# Ajustement des hyperparamètres d'entraînement
taux_apprentissage = 0.1f0  # Idéal pour la Softmax Cross-Entropy
epoches = 5

println("🏋️ Début de l'entraînement de NeuroDSL v5 sur le VRAI MNIST (468 batchs/époque)...")
println("--------------------------------------------------------------------------------")

for epoch in 1:epoches
    # Exécution du pipeline asynchrone sur l'ensemble des batchs réels
    train_mnist_pipeline!(graphe_vrai_mnist, x_vrai, y_vrai, taux_apprentissage)

    # Identification et lecture dynamique du dernier slot actif pour le monitoring
    dernier_slot = (length(x_vrai) % 2 == 0) ? graphe_vrai_mnist.batch_B : graphe_vrai_mnist.batch_A

    loss_e = Array(dernier_slot[:loss].value)[1]
    acc_e = calculer_accuracy(dernier_slot)

    println("▶️ ÉPOQUE $epoch / $epoches | Loss finale : $(round(loss_e, digits=4)) | Précision Lot : $(round(acc_e, digits=2))%")
end

println("--------------------------------------------------------------------------------")
println("🎉 Entraînement terminé ! Ton moteur réactif asynchrone a convergé.")

✅ Pas 2 réussi : Poids initialisés et Streams CUDA (A et B) alloués séparément.
🏋️ Début de l'entraînement de NeuroDSL v5 sur le VRAI MNIST (468 batchs/époque)...
--------------------------------------------------------------------------------
✅ Pas 4 mis à jour : Signature assouplie et prête.
▶️ ÉPOQUE 1 / 5 | Loss finale : 0.0811 | Précision Lot : 99.22%
✅ Pas 4 mis à jour : Signature assouplie et prête.
▶️ ÉPOQUE 2 / 5 | Loss finale : 0.0514 | Précision Lot : 99.22%
✅ Pas 4 mis à jour : Signature assouplie et prête.
▶️ ÉPOQUE 3 / 5 | Loss finale : 0.0394 | Précision Lot : 99.22%
✅ Pas 4 mis à jour : Signature assouplie et prête.
▶️ ÉPOQUE 4 / 5 | Loss finale : 0.0336 | Précision Lot : 100.0%
✅ Pas 4 mis à jour : Signature assouplie et prête.
▶️ ÉPOQUE 5 / 5 | Loss finale : 0.0297 | Précision Lot : 100.0%
--------------------------------------------------------------------------------
🎉 Entraînement terminé ! Ton moteur réactif asynchrone a convergé.


In [9]:
function preparer_test_mnist(B::Int)
    # Chargement du split de test (10 000 images)
    test_x, test_y = MNIST(split=:test)[:]
    num_samples = size(test_x, 3)

    # Aplatissement et normalisation
    X_flat = zeros(Float32, num_samples, 784)
    for i in 1:num_samples
        X_flat[i, :] = vec(test_x[:, :, i])
    end

    # Encodage One-Hot
    Y_onehot = zeros(Float32, num_samples, 10)
    for i in 1:num_samples
        label = test_y[i]
        Y_onehot[i, label + 1] = 1.0f0
    end

    # Découpage en mini-batchs pour le GPU
    x_batches = [CuArray(X_flat[i:min(i+B-1, num_samples), :]) for i in 1:B:num_samples]
    y_batches = [CuArray(Y_onehot[i:min(i+B-1, num_samples), :]) for i in 1:B:num_samples]

    # Garder des tailles fixes
    if size(x_batches[end], 1) != B
        pop!(x_batches)
        pop!(y_batches)
    end

    println("📥 VRAI MNIST Test prêt : $(length(x_batches)) batchs de taille $B en VRAM.")
    return x_batches, y_batches
end

# Chargement avec la même taille de batch (128)
x_test, y_test = preparer_test_mnist(128)

📥 VRAI MNIST Test prêt : 78 batchs de taille 128 en VRAM.


(CuArray{Float32, 2, CUDACore.DeviceMemory}[[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]  …  [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 

In [10]:
function evaluer_modele_pipeline(graph::MnistParallelGraph, x_data::Vector{<:CuArray}, y_data::Vector{<:CuArray})
    num_batches = length(x_data)
    total_correct = 0
    total_images = 0

    # Amorçage : Premier forward sur le Slot A
    execute_forward_graph_mnist!(graph.batch_A, graph.weights, x_data[1], y_data[1])

    # Boucle asynchrone entrelacée
    for i in 2:num_batches
        current_slot = (i % 2 == 0) ? graph.batch_A : graph.batch_B
        next_slot    = (i % 2 == 0) ? graph.batch_B : graph.batch_A

        X_next = x_data[i]
        Y_next = y_data[i]

        # 1. Pendant que le CPU récupère les résultats du slot actuel (bloquant relâché dès que prêt)
        pred_cpu = Array(current_slot[:pred].value)
        y_cpu = Array(current_slot[:y].value)

        # Calcul des prédictions correctes sur le CPU
        for j in 1:size(pred_cpu, 1)
            if argmax(pred_cpu[j, :]) == argmax(y_cpu[j, :])
                total_correct += 1
            end
            total_images += 1
        end

        # 2. On lance immédiatement le Forward du batch suivant en arrière-plan sur l'autre stream
        tache_forward = @async begin
            execute_forward_graph_mnist!(next_slot, graph.weights, X_next, Y_next)
        end

        wait(tache_forward)
    end

    # Traitement du tout dernier batch restant dans le pipeline
    dernier_slot = (num_batches % 2 == 0) ? graph.batch_B : graph.batch_A
    pred_cpu = Array(dernier_slot[:pred].value)
    y_cpu = Array(dernier_slot[:y].value)
    for j in 1:size(pred_cpu, 1)
        if argmax(pred_cpu[j, :]) == argmax(y_cpu[j, :])
            total_correct += 1
        end
        total_images += 1
    end

    accuracy_globale = (total_correct / total_images) * 100
    return accuracy_globale
end

evaluer_modele_pipeline (generic function with 1 method)

# Resultats

In [11]:
println("🔍 Évaluation en cours sur l'ensemble de Test indépendant...")
accuracy_finale = evaluer_modele_pipeline(graphe_vrai_mnist, x_test, y_test)

println("\n========================================================")
println("🎯 ACCURACY GLOBALE SUR LE JEU DE TEST : $(round(accuracy_finale, digits=2)) %")
println("========================================================")

🔍 Évaluation en cours sur l'ensemble de Test indépendant...

🎯 ACCURACY GLOBALE SUR LE JEU DE TEST : 95.38 %
